# **SVM NOTEBOOK**

In [1]:
# Import

import json
import sys
from pathlib import Path
# Import preprocessing module from backend
sys.path.append("..")
import backend.app.ml.preprocessing as preprocessing
import importlib

import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
classification_report,
confusion_matrix,
accuracy_score,
precision_score,
recall_score,
f1_score,
roc_auc_score,
roc_curve
)



sns.set_theme(style="whitegrid")

In [ ]:
# Read data and run preprocessing pipeline

raw_df = pd.read_csv("../dataset/all_zones_complete.csv")

df_processed = prepare_data(
raw_df,
include_lags=True
)

df_processed = df_processed.dropna().reset_index(drop=True)

print(f"Processed data form (all zones): {df_processed.shape}")

df_processed.head(3)

FileNotFoundError: [Errno 2] No such file or directory: '../dataset/all_zones_complete_2025.csv'

In [ ]:
# Create target variable

HOURS_TO_SELECT = 6 # The 6 cheapest hours per day and zone

In [ ]:
# Create date column for grouping

df_processed["date"] = df_processed["timestamp_local"].dt.date

In [ ]:
# Calculate price rank per zone and day

df_processed["price_rank"] = (
df_processed
.groupby(["zone", "date"])["spot_price_eur_mwh"]
.rank(
method="min",
ascending=True
)
)

In [ ]:
# Mark the 6 cheapest hours as optimal

df_processed["optimal_timme"] = (
df_processed["price_rank"] <= HOURS_TO_SELECT
).astype(int)

print("Total distribution of the target variable for all zones:")
print(df_processed["optimal_timme"].value_counts())

In [ ]:
# One-hot encode electricity zones

df_processed = pd.get_dummies(
df_processed,
columns=["zone"],
drop_first=False
)

In [ ]:
# Choose features and target

feature_cols = [
"temperature_c",
"wind_speed_kmh",
"rain_mm",
"hour",
"day_of_week",
"month",
"is_weekend",
"price_lag_24",
"price_lag_48",
"price_lag_168",
"zone_SE1",
"zone_SE2",
"zone_SE3",
"zone_SE4"
]

X = df_processed[feature_cols]
y = df_processed["optimal_timme"]

In [ ]:
# Train / Validation / Test split
# 70% Train, 15% Validation, 15% Test

X_train_val, X_test, y_train_val, y_test = train_test_split(
X,
y,
test_size=0.15,
random_state=42,
stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
X_train_val,
y_train_val,
test_size=0.1764,
random_state=42,
stratify=y_train_val
)

print(f"Traningset: {X_train.shape[0]} rader")
print(f"Validationset: {X_val.shape[0]} rader")
print(f"Testset: {X_test.shape[0]} rader")

In [ ]:
# Scaling

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Hyperparameter optimization with GridSearchCV

param_grid = {
"C": [0.1, 1, 10],
"gamma": ["scale", 0.1],
"kernel": ["rbf"]
}

svm = SVC(
class_weight="balanced",
probability=True,
random_state=42
)

grid_search = GridSearchCV(
estimator=svm,
param_grid=param_grid,
scoring="f1",
cv=3,
n_jobs=-1,
verbose=1
)

grid_search.fit(
X_train_scaled,
y_train
)

print("\nBest params:")
print(grid_search.best_params_)

best_model = grid_search.best_estimator_

In [ ]:
# Evaluate model

def evaluate_model(
    model,
    X_data,
    y_data,
    dataset_name="Test"
):
    y_pred = model.predict(X_data)
    y_prob = model.predict_proba(X_data)[:, 1]

    print(f"--- Evaluation on {dataset_name} ---")

    print(f"Accuracy:  {accuracy_score(y_data, y_pred):.4f}")
    print(f"Precision: {precision_score(y_data, y_pred):.4f}")
    print(f"Recall:    {recall_score(y_data, y_pred):.4f}")
    print(f"F1-score:  {f1_score(y_data, y_pred):.4f}")
    print(f"ROC-AUC:   {roc_auc_score(y_data, y_prob):.4f}\n")

    print("Classification Report:")
    print(classification_report(y_data, y_pred))

In [ ]:
# Evaluate on validation set

evaluate_model(
best_model,
X_val_scaled,
y_val,
dataset_name="Validation"
)

In [ ]:
# Evaluate on test set

evaluate_model(
best_model,
X_test_scaled,
y_test,
dataset_name="Testset"
)

In [ ]:
# Visualization – Confusion Matrix

y_pred_test = best_model.predict(X_test_scaled)

cm = confusion_matrix(
y_test,
y_pred_test
)

plt.figure(figsize=(7, 5))

sns.heatmap(
cm,
annot=True,
fmt="d",
cmap="Blues",
cbar=False
)

plt.title("Confusion Matrix (Testset)")
plt.xlabel("Predicted class")
plt.ylabel("Real class")

plt.tight_layout()
plt.show()

In [ ]:
# Visualization – ROC Curve

y_prob_test = best_model.predict_proba(
X_test_scaled
)[:, 1]

fpr, tpr, thresholds = roc_curve(
y_test,
y_prob_test
)

auc_val = roc_auc_score(
y_test,
y_prob_test
)

plt.figure(figsize=(7, 5))

plt.plot(
fpr,
tpr,
label=f"SVM (AUC = {auc_val:.3f})",
color="darkorange",
lw=2
)

plt.plot(
[0, 1],
[0, 1],
"k--",
lw=2
)

plt.title("ROC-curv")
plt.xlabel("False positiv rate (FPR)")
plt.ylabel("True positiv rate (TPR)")
plt.legend(loc="lower right")

plt.tight_layout()
plt.show()

In [ ]:
# Business simulation on test set

df_test_eval = df_processed.loc[X_test.index].copy()

df_test_eval["predicted_optimal"] = y_pred_test

In [ ]:
# 1. Average price selected by the model

model_avg_price = (
df_test_eval[
df_test_eval["predicted_optimal"] == 1
]["spot_price_eur_mwh"]
.mean()
)

In [ ]:
# 2. Average price for actual optimal hours

actual_best_avg_price = (
df_test_eval[
df_test_eval["optimal_timme"] == 1
]["spot_price_eur_mwh"]
.mean()
)

In [ ]:
# 3. Average price across the complete test set

total_avg_price = (
df_test_eval["spot_price_eur_mwh"]
.mean()
)

In [ ]:
# 4. Average price for non-optimal hours
worst_avg_price = (
df_test_eval[
df_test_eval["optimal_timme"] == 0
]["spot_price_eur_mwh"]
.mean()
)

print("--- Business Price Simulation (Testset) ---")

print(
f"The model's selected average price: "
f"{model_avg_price:.2f} EUR/MWh"
)

print(
f"Theoretically best average price: "
f"{actual_best_avg_price:.2f} EUR/MWh"
)

print(
f"The total test set's average price: "
f"{total_avg_price:.2f} EUR/MWh"
)

print(
f"Average price for non-optimal hours: "
f"{worst_avg_price:.2f} EUR/MWh"
)

print(
f"\nThe model saves about "
f"{total_avg_price - model_avg_price:.2f} EUR/MWh "
f"on average compared to not optimizing at all!"
)


In [ ]:
# Save trained model, scaler and feature columns

models_dir = Path("../models_bin")
models_dir.mkdir(exist_ok=True)

In [ ]:
# Save SVM model

joblib.dump(
best_model,
models_dir / "svm_optimal_classifier.joblib"
)

In [ ]:
# Save scaler

joblib.dump(
scaler,
models_dir / "scaler.joblib"
)

In [ ]:
# Save feature columns

with open(
    models_dir / "feature_cols.json",
    "w"
) as f:
    json.dump(
        feature_cols,
        f,
        indent=4
    )

print("\nModel, scaler, and feature columns have been saved in:")
print(models_dir)